# 03 - Transfer Learning Classification

This notebook trains a MobileNetV2 classifier with ImageNet weights, first with the backbone frozen and then with upper-layer fine-tuning.

## Google Colab Git Setup

Run the next cell only when using Google Colab. Set `REPO_URL` to your GitHub repository URL, then the cell clones or pulls the repository into `/content/PetVision-DeepLearning` and installs dependencies. If you run locally, skip it.


In [ ]:
# Colab-only Git setup. Skip this cell when running locally.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/andri-10/computer_vision.git"
PROJECT_DIR = Path('/PetVision-DeepLearning')

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    if 'YOUR_USERNAME' in REPO_URL:
        raise ValueError('Replace REPO_URL with your GitHub repository URL before running this cell.')
    if PROJECT_DIR.exists():
        subprocess.check_call(['git', '-C', str(PROJECT_DIR), 'pull'])
    else:
        subprocess.check_call(['git', 'clone', REPO_URL, str(PROJECT_DIR)])
    os.chdir(PROJECT_DIR)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
    print('Colab project root:', os.getcwd())
else:
    print('Not running in Google Colab. Continue with the local setup cells below.')


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

for path in [
    "models",
    "results/classification",
    "results/segmentation",
    "results/gradcam",
    "results/figures",
]:
    (PROJECT_ROOT / path).mkdir(parents=True, exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from src.data_loader import get_splits, configure_for_performance, save_label_mapping
from src.preprocessing import preprocess_transfer_classification
from src.models_classification import build_transfer_classifier, compile_classifier, unfreeze_for_fine_tuning
from src.training import standard_callbacks
from src.evaluation import collect_predictions, save_classification_outputs, top_k_accuracy, measure_inference_time
from src.visualization import plot_training_history, plot_confusion_matrix

In [ ]:
BATCH_SIZE = 32
HEAD_EPOCHS = 8
FINE_TUNE_EPOCHS = 8
BACKBONE = "MobileNetV2"

train_raw, val_raw, test_raw, info, label_names = get_splits()
save_label_mapping(label_names, PROJECT_ROOT / "results/figures/label_mapping.json")

prep = lambda example: preprocess_transfer_classification(example, backbone="mobilenet_v2")
train_ds = configure_for_performance(train_raw.map(prep, num_parallel_calls=tf.data.AUTOTUNE), BATCH_SIZE, shuffle=True)
val_ds = configure_for_performance(val_raw.map(prep, num_parallel_calls=tf.data.AUTOTUNE), BATCH_SIZE)
test_ds = configure_for_performance(test_raw.map(prep, num_parallel_calls=tf.data.AUTOTUNE), BATCH_SIZE)

In [ ]:
model, base_model = build_transfer_classifier(input_shape=(224, 224, 3), num_classes=len(label_names), backbone_name=BACKBONE)
compile_classifier(model, learning_rate=1e-3)
model.summary()

In [ ]:
head_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=HEAD_EPOCHS,
    callbacks=standard_callbacks(PROJECT_ROOT / "models/best_classifier.keras", patience=4),
)
plot_training_history(head_history, PROJECT_ROOT / "results/classification/transfer_head_training_curves.png", "Transfer Head")

In [ ]:
unfreeze_for_fine_tuning(base_model)
compile_classifier(model, learning_rate=1e-5)

fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS,
    callbacks=standard_callbacks(PROJECT_ROOT / "models/best_classifier.keras", patience=4),
)
plot_training_history(fine_tune_history, PROJECT_ROOT / "results/classification/transfer_training_curves.png", "Transfer Fine-tuning")

In [ ]:
best_model = tf.keras.models.load_model(PROJECT_ROOT / "models/best_classifier.keras")
test_metrics = best_model.evaluate(test_ds, verbose=1)
print(dict(zip(best_model.metrics_names, test_metrics)))

y_true, y_pred, y_prob = collect_predictions(best_model, test_ds)
transfer_top3 = top_k_accuracy(y_true, y_prob, k=3)
print("Top-3 accuracy:", transfer_top3)
save_classification_outputs(y_true, y_pred, label_names, PROJECT_ROOT / "results/classification/transfer")
plot_confusion_matrix(y_true, y_pred, label_names, PROJECT_ROOT / "results/classification/transfer_confusion_matrix.png")

In [ ]:
# Model comparison table. Fill baseline values after running notebook 02.
sample_batch = next(iter(test_ds))[0]
inference_ms = measure_inference_time(best_model, sample_batch, runs=10)
model_size_mb = (PROJECT_ROOT / "models/best_classifier.keras").stat().st_size / (1024 * 1024)
comparison = pd.DataFrame([
    {"model": "Baseline CNN", "input_size": "128x128", "test_accuracy": None, "top_3_accuracy": None, "model_size_mb": None, "inference_ms_per_batch": None},
    {"model": BACKBONE, "input_size": "224x224", "test_accuracy": float(np.mean(y_true == y_pred)), "top_3_accuracy": transfer_top3, "model_size_mb": model_size_mb, "inference_ms_per_batch": inference_ms},
])
comparison.to_csv(PROJECT_ROOT / "results/classification/model_comparison_table.csv", index=False)
comparison